In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo","catalog_dev")
dbutils.widgets.text("esquema_source","bronze")
dbutils.widgets.text("esquema_sink","silver")
dbutils.widgets.text("tabla_bronze","ecommerce_sales_prediction_bronze")
dbutils.widgets.text("tabla_silver","ecommerce_sales_prediction_silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")
tabla_bronze = dbutils.widgets.get("tabla_bronze")
tabla_silver = dbutils.widgets.get("tabla_silver")

In [0]:
df_bronze = spark.table(f"{catalogo}.{esquema_source}.{tabla_bronze}")

In [0]:
# Se asume que el descuento es por unidad, por tanto no puede superar el precio
df_silver = df_bronze.filter(
    (col("Discount") >= 0) &
    (col("Discount") <= col("Price"))
)

In [0]:
df_silver = df_silver.withColumn(
    "Net_Unit_Price",
    col("Price") - col("Discount")
)

df_silver = df_silver.withColumn(
    "Revenue",
    col("Net_Unit_Price") * col("Units_Sold")
)

df_silver = df_silver.withColumn(
    "Discount_Rate",
    col("Discount") / col("Price")
)

df_silver = df_silver.withColumn(
    "Discount_Level",
    when(col("Discount_Rate") < 0.05, "Low")
    .when(col("Discount_Rate") < 0.15, "Medium")
    .otherwise("High")
)

df_silver = (
    df_silver
    .withColumn("Year", year("Date"))
    .withColumn("Month", month("Date"))
    .withColumn("Quarter", quarter("Date"))
)


In [0]:
df_silver.limit(10).display()

In [0]:
df_silver.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.{tabla_silver}")